# OGSOD-1.0 YOLOv8s-OBB + P2 + ECA Colab Workflow

This notebook keeps training on the T4 GPU. The local project is used only for code, conversion, and checks.

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > T4 GPU before continuing.')

In [ ]:
from pathlib import Path
import os

REPO_URL = 'https://github.com/Cranzz/ogsod-obb-p2-eca.git'
REPO_DIR = Path('/content/OGSOD-1.0_SAR_YOLOv8s-OBB-P2-ECA')

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only

%cd {REPO_DIR}
!python -m pip install -q 'ultralytics==8.4.95'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/OGSOD')
SOURCE_ROOT = DRIVE_ROOT / 'OGSOD-1.0'
OBB_LABEL_ROOT = DRIVE_ROOT / 'OGSOD-1.0-obb_label'
LEGACY_OBB_LABEL_ROOT = DRIVE_ROOT / 'OGSOD-1.0(obb_label) '
OBB_LABEL_ZIP = DRIVE_ROOT / 'OGSOD-1.0-obb_label.zip'
LOCAL_OBB_ROOT = Path('/content/OGSOD-1.0-obb_label')
if OBB_LABEL_ZIP.exists():
    LOCAL_OBB_ROOT.mkdir(parents=True, exist_ok=True)
    !unzip -q -o '{OBB_LABEL_ZIP}' -d '{LOCAL_OBB_ROOT}'
    candidates = [LOCAL_OBB_ROOT / 'OGSOD-1.0(obb_label) ', LOCAL_OBB_ROOT / 'OGSOD-1.0-obb_label']
    OBB_LABEL_ROOT = next(path for path in candidates if path.exists())
elif not OBB_LABEL_ROOT.exists() and LEGACY_OBB_LABEL_ROOT.exists():
    OBB_LABEL_ROOT = LEGACY_OBB_LABEL_ROOT
DERIVED_ROOT = Path('/content/OGSOD-1.0-yolo-obb')

for path in (SOURCE_ROOT, OBB_LABEL_ROOT):
    if not path.exists():
        raise FileNotFoundError(f'Upload the dataset to Google Drive first: {path}')

!python scripts/convert_ogsod_obb.py \
  --source-root '{SOURCE_ROOT}' \
  --obb-root '{OBB_LABEL_ROOT}' \
  --output '{DERIVED_ROOT}' \
  --val-fraction 0.1 \
  --seed 42 \
  --images-mode symlink

DATA_YAML = DERIVED_ROOT / 'data.yaml'
print(DATA_YAML)

In [ ]:
import random

def write_subset(source_list, output_list, count, seed):
    lines = [line for line in Path(source_list).read_text().splitlines() if line.strip()]
    random.Random(seed).shuffle(lines)
    Path(output_list).write_text('\n'.join(lines[:count]) + '\n')
    return len(lines[:count])

train_count = write_subset(DERIVED_ROOT / 'train.txt', DERIVED_ROOT / 'smoke_train.txt', 100, 42)
val_count = write_subset(DERIVED_ROOT / 'val.txt', DERIVED_ROOT / 'smoke_val.txt', 30, 42)
SMOKE_YAML = DERIVED_ROOT / 'smoke_data.yaml'
SMOKE_YAML.write_text(
    f'path: {DERIVED_ROOT}\ntrain: smoke_train.txt\nval: smoke_val.txt\ntest: test.txt\n'
    'names:\n  0: bridge\n  1: harbor\n  2: storage_tank\n'
)
print('smoke train/val:', train_count, val_count)
print(SMOKE_YAML)

In [ ]:
# First cloud test: only 100 train images, 30 validation images, and one epoch.
!python scripts/train_obb.py \
  --data {SMOKE_YAML} \
  --experiment baseline \
  --epochs 1 \
  --imgsz 640 \
  --batch 8 \
  --workers 2 \
  --device 0 \
  --project /content/runs \
  --name smoke_baseline \
  --eval-split val

In [ ]:
EXPERIMENT = 'baseline'  # baseline, p2, eca, p2_eca
EPOCHS = 100
IMGSZ = 640
BATCH = 16
SEED = 42
RUN_NAME = f'yolov8s_obb_ogsod_{EXPERIMENT}_seed{SEED}'

!python scripts/train_obb.py \
  --data {DATA_YAML} \
  --experiment {EXPERIMENT} \
  --epochs {EPOCHS} \
  --imgsz {IMGSZ} \
  --batch {BATCH} \
  --workers 4 \
  --device 0 \
  --seed {SEED} \
  --project /content/runs \
  --name {RUN_NAME} \
  --eval-split test

In [ ]:
# Persist logs, plots, and weights outside the ephemeral Colab runtime.
from shutil import copytree
RUN_DIR = Path('/content/runs') / RUN_NAME
DRIVE_RUN_DIR = DRIVE_ROOT / 'runs' / RUN_NAME
if DRIVE_RUN_DIR.exists():
    raise FileExistsError(f'Refusing to overwrite existing run: {DRIVE_RUN_DIR}')
copytree(RUN_DIR, DRIVE_RUN_DIR)
print(DRIVE_RUN_DIR)